# Gold - dim_product

This notebook loads `amperedrive_giga.gold.dim_product` from `amperedrive_giga.silver.dim_products`, one row per EV vehicle or component (`product_id`). Load pattern: Incremental Merge (SCD Type 1), using a watermark on `silver.updated_at` compared with `MAX(gold.source_updated_at)`. DQ-G01 (non-negative cost and price) stops the load if any incoming row fails.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS amperedrive_giga.gold.dim_product (
  product_sk               BIGINT         NOT NULL COMMENT 'PK - xxhash64(product_id)',
  product_id               INT            NOT NULL COMMENT 'NK',
  plant_id                 INT            NOT NULL COMMENT 'FK - plant natural key',
  product_name             STRING         NOT NULL,
  product_category         STRING         NOT NULL,
  is_finished_vehicle      BOOLEAN        NOT NULL,
  material_cost_usd        DECIMAL(38,18) NOT NULL,
  factory_output_cost_usd  DECIMAL(38,18) NOT NULL,
  standard_unit_margin_usd DECIMAL(38,18) NOT NULL,
  standard_cycle_time_secs INT            NOT NULL,
  source_inserted_at       TIMESTAMP      NOT NULL,
  source_updated_at        TIMESTAMP      NOT NULL,
  gold_load_ts             TIMESTAMP      NOT NULL,
  CONSTRAINT pk_dim_product PRIMARY KEY (product_sk)
)
COMMENT 'Gold product dimension (SCD Type 1) sourced from silver.dim_products'

In [0]:
%sql
-- Watermark = latest source_updated_at already in gold (falls back to 1900-01-01 on first run)
DECLARE OR REPLACE VARIABLE v_watermark TIMESTAMP;
SET VAR v_watermark = (
  SELECT COALESCE(MAX(source_updated_at), TIMESTAMP'1900-01-01 00:00:00')
  FROM amperedrive_giga.gold.dim_product
);
SELECT v_watermark AS watermark;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_src_dim_product AS
SELECT
  xxhash64(CAST(product_id AS STRING))                                  AS product_sk,
  product_id,
  plant_id,
  TRIM(product_name)                                                    AS product_name,
  TRIM(product_category)                                                AS product_category,
  CASE WHEN product_category = 'Finished Vehicle' THEN TRUE ELSE FALSE END AS is_finished_vehicle,
  material_cost_usd,
  factory_output_cost_usd,
  CAST(factory_output_cost_usd - material_cost_usd AS DECIMAL(38,18))  AS standard_unit_margin_usd,
  standard_cycle_time_secs,
  inserted_at                                                           AS source_inserted_at,
  updated_at                                                            AS source_updated_at
FROM amperedrive_giga.silver.dim_products
WHERE product_id IS NOT NULL
  AND updated_at > v_watermark
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY updated_at DESC NULLS LAST) = 1;

SELECT COUNT(*) AS incoming_rows FROM vw_src_dim_product;

In [0]:
%sql
-- DQ-G01: material_cost_usd > 0 AND factory_output_cost_usd >= material_cost_usd -> Quarantine / Fail Pipeline
SELECT
  COUNT(*) AS dq_g01_violations,
  CASE WHEN COUNT(*) > 0
       THEN raise_error(CONCAT('DQ-G01 failed: ', COUNT(*), ' dim_product rows have invalid cost/price'))
  END AS dq_g01_status
FROM vw_src_dim_product
WHERE NOT (material_cost_usd > 0 AND factory_output_cost_usd >= material_cost_usd)
   OR material_cost_usd IS NULL
   OR factory_output_cost_usd IS NULL

In [0]:
%sql
MERGE INTO amperedrive_giga.gold.dim_product AS t
USING vw_src_dim_product AS s
ON t.product_id = s.product_id
WHEN MATCHED AND (
     NOT (t.plant_id                 <=> s.plant_id)
  OR NOT (t.product_name             <=> s.product_name)
  OR NOT (t.product_category         <=> s.product_category)
  OR NOT (t.material_cost_usd        <=> s.material_cost_usd)
  OR NOT (t.factory_output_cost_usd  <=> s.factory_output_cost_usd)
  OR NOT (t.standard_cycle_time_secs <=> s.standard_cycle_time_secs)
  OR NOT (t.source_inserted_at       <=> s.source_inserted_at)
  OR NOT (t.source_updated_at        <=> s.source_updated_at)
) THEN UPDATE SET
  t.plant_id                 = s.plant_id,
  t.product_name             = s.product_name,
  t.product_category         = s.product_category,
  t.is_finished_vehicle      = s.is_finished_vehicle,
  t.material_cost_usd        = s.material_cost_usd,
  t.factory_output_cost_usd  = s.factory_output_cost_usd,
  t.standard_unit_margin_usd = s.standard_unit_margin_usd,
  t.standard_cycle_time_secs = s.standard_cycle_time_secs,
  t.source_inserted_at       = s.source_inserted_at,
  t.source_updated_at        = s.source_updated_at,
  t.gold_load_ts             = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
  product_sk, product_id, plant_id, product_name, product_category, is_finished_vehicle,
  material_cost_usd, factory_output_cost_usd, standard_unit_margin_usd, standard_cycle_time_secs,
  source_inserted_at, source_updated_at, gold_load_ts
) VALUES (
  s.product_sk, s.product_id, s.plant_id, s.product_name, s.product_category, s.is_finished_vehicle,
  s.material_cost_usd, s.factory_output_cost_usd, s.standard_unit_margin_usd, s.standard_cycle_time_secs,
  s.source_inserted_at, s.source_updated_at, CURRENT_TIMESTAMP()
)

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM amperedrive_giga.gold.dim_product)                     AS gold_rows,
  (SELECT COUNT(DISTINCT product_id) FROM amperedrive_giga.silver.dim_products) AS silver_distinct_products,
  (SELECT COUNT(*) FROM amperedrive_giga.gold.dim_product WHERE is_finished_vehicle) AS finished_vehicles,
  assert_true(
    (SELECT COUNT(*) - COUNT(DISTINCT product_sk) FROM amperedrive_giga.gold.dim_product) = 0,
    'dim_product: duplicate product_sk') AS dq_unique_sk,
  assert_true(
    (SELECT COUNT(*) FROM amperedrive_giga.gold.dim_product
      WHERE NOT (material_cost_usd > 0 AND factory_output_cost_usd >= material_cost_usd)) = 0,
    'DQ-G01 failed on gold.dim_product') AS dq_g01